In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## Overview

BB84 with an eavesdropper, Eve. Two attacks are simulated:

1. **Full intercept-and-resend** - Eve catches every qubit, measures in a random basis, then forwards a fresh qubit encoding her measured bit. Expected sifted-key error: ~25%.
2. **Partial intercept** - Eve only attacks a fraction $p$ of the qubits and leaves the rest untouched. She gets less information about the key, but also causes less disturbance. Expected sifted-key error: ~$0.25 \cdot p$.

Alice and Bob will proceed normally as they don't know Eve is there. Eve can't avoid disturbing the qubits, and that disturbance shows up in the error check. The partial-intercept attack shows how much she can get away with before crossing the threshold.

All random choices come from quantum measurements of $H|0\rangle$. Sections are labelled [ALICE], [EVE], [BOB], [CLASSICAL].

Steps:
1. Alice picks random bits and bases
2. Eve picks random interception bases
3. Bob picks random measurement bases
4. Alice encodes → Eve intercepts and resends → Bob measures
5. Sifting on Alice's and Bob's bases (Eve's bases stay hidden)
6. Error check on a sample of sifted bits
7. Breakdown of where Eve's disturbance came from
8. Partial-intercept attack: sweep $p$ and find the threshold-crossing point

## Quantum random bit generator

Prepare $|+\rangle$, measure in Z, get 0 or 1 with probability $\tfrac{1}{2}$. Shared by all three parties.

In [3]:
def quantum_random_bits(n: int) -> list[int]:
    """Return n random bits from measuring n independent |+> qubits."""
    out = []
    for _ in range(n):
        qc = QuantumCircuit(1, 1)
        qc.h(0)
        qc.measure(0, 0)
        tqc = transpile(qc, backend)
        counts = backend.run(tqc, shots=1).result().get_counts()
        out.append(int(next(iter(counts))))
    return out

## Alice: encoding

Two bases, two bit values, four possible states:

| Bit | Z basis (0) | X basis (1) |
|-----|-------------|-------------|
| 0   | $\lvert 0\rangle$ | $\lvert +\rangle$ |
| 1   | $\lvert 1\rangle$ | $\lvert -\rangle$ |

Encoding rule: start in $\lvert 0\rangle$, apply $X$ if the bit is 1, then apply $H$ if the basis is X.

In [4]:
def alice_encode(bit: int, basis: int) -> QuantumCircuit:
    """Build the qubit Alice sends for (bit, basis). No measurement."""
    qc = QuantumCircuit(1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

## Eve: intercept and resend

For every qubit Eve decides to attack:
1. She picks a basis (Z or X).
2. She measures Alice's qubit in that basis, getting a bit. This collapses the state.
3. She prepares a fresh qubit encoding that bit in her basis and forwards it.

She can't clone the qubit (no-cloning theorem), so step 2 is unavoidable. Whenever her basis disagrees with Alice's, her measurement randomises the state. That's what shows up in the error check.

In [5]:
def eve_attack(received: QuantumCircuit, basis: int) -> tuple[int, QuantumCircuit]:
    """Measure the incoming qubit in `basis`; return (measured_bit, fresh_qubit_for_bob)."""
    qc = QuantumCircuit(1, 1)
    qc.compose(received, inplace=True)
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=1).result().get_counts()
    measured = int(next(iter(counts)))
    return measured, alice_encode(measured, basis)

## Bob: measurement

Bob measures whatever qubit he receives. He doesn't know Eve has tampered with it.

In [6]:
def bob_measure(received: QuantumCircuit, basis: int) -> int:
    """Measure the incoming qubit in Bob's chosen basis; return 0 or 1."""
    qc = QuantumCircuit(1, 1)
    qc.compose(received, inplace=True)
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=1).result().get_counts()
    return int(next(iter(counts)))

## Attack 1: full intercept-and-resend

In [7]:
N = 100    # qubits transmitted
THRESHOLD = 0.15   # abort if observed error rate exceeds
                   # no attacker: ~0%, full intercept: ~25%, partial p: ~0.25*p

print("=" * 56)
print("  BB84: Eve, full intercept-and-resend")
print("=" * 56)
print(f"N = {N}, abort threshold = {THRESHOLD:.0%}\n")

  BB84: Eve, full intercept-and-resend
N = 100, abort threshold = 15%



In [8]:
# Step 1: Alice's bits and bases
print("[ALICE] picking bits and bases")
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

print(f"[ALICE]  bits  (first 20): {alice_bits[:20]}")
print(f"[ALICE]  bases (first 20): {alice_bases[:20]}   (0=Z, 1=X)")

[ALICE] picking bits and bases
[ALICE]  bits  (first 20): [0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1]
[ALICE]  bases (first 20): [0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1]   (0=Z, 1=X)


In [9]:
# Step 2: Eve's interception bases
print("[EVE] picking interception bases")
eve_bases = quantum_random_bits(N)

print(f"[EVE]    bases (first 20): {eve_bases[:20]}   (0=Z, 1=X)")

[EVE] picking interception bases
[EVE]    bases (first 20): [0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0]   (0=Z, 1=X)


In [10]:
# Step 3: Bob's measurement bases
print("[BOB] picking measurement bases")
bob_bases = quantum_random_bits(N)

print(f"[BOB]    bases (first 20): {bob_bases[:20]}   (0=Z, 1=X)")

[BOB] picking measurement bases
[BOB]    bases (first 20): [1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1]   (0=Z, 1=X)


In [11]:
# Step 4: Quantum channel with Eve in the middle
print("[CHANNEL] Alice encodes -> Eve intercepts + resends -> Bob measures")

eve_bits = []   # Eve's measured bits (her private info)
bob_bits = []   # Bob's measured bits

for i in range(N):
    sent_by_alice           = alice_encode(alice_bits[i], alice_bases[i])
    eve_measured, forwarded = eve_attack(sent_by_alice, eve_bases[i])
    eve_bits.append(eve_measured)
    bob_bits.append(bob_measure(forwarded, bob_bases[i]))

print(f"[EVE]    measured (first 20): {eve_bits[:20]}")
print(f"[BOB]    measured (first 20): {bob_bits[:20]}")

[CHANNEL] Alice encodes -> Eve intercepts + resends -> Bob measures
[EVE]    measured (first 20): [0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0]
[BOB]    measured (first 20): [0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0]


In [12]:
# Step 5: Sifting (Alice's vs Bob's bases; Eve's stay hidden)
print("[CLASSICAL] Alice and Bob compare bases, drop the mismatches")

keep     = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
sifted_a = [alice_bits[i] for i in keep]
sifted_b = [bob_bits[i]   for i in keep]

print(f"[SIFT]   kept {len(keep)} / {N} positions")
print(f"[ALICE]  sifted (first 20): {sifted_a[:20]}")
print(f"[BOB]    sifted (first 20): {sifted_b[:20]}")

[CLASSICAL] Alice and Bob compare bases, drop the mismatches
[SIFT]   kept 53 / 100 positions
[ALICE]  sifted (first 20): [1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1]
[BOB]    sifted (first 20): [1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1]


In [13]:
# Step 6: Error check
print("[CLASSICAL] revealing a sample to estimate the error rate")

sample_n = max(1, len(sifted_a) // 3)
errs     = sum(1 for a, b in zip(sifted_a[:sample_n], sifted_b[:sample_n]) if a != b)
err_rate = errs / sample_n

print(f"\n[CHECK]  sample size : {sample_n}")
print(f"[CHECK]  mismatches  : {errs}")
print(f"[CHECK]  error rate  : {err_rate:.2%}")
print(f"[CHECK]  threshold   : {THRESHOLD:.2%}")
print(f"[CHECK]  theory      : ~25% under full intercept-and-resend")

if err_rate > THRESHOLD:
    print(f"\n[ALERT] {err_rate:.2%} > {THRESHOLD:.2%}, eavesdropper detected, aborting")
else:
    print("\n[OK]    error rate within threshold (unusual for full Eve, small-sample noise)")

[CLASSICAL] revealing a sample to estimate the error rate

[CHECK]  sample size : 17
[CHECK]  mismatches  : 4
[CHECK]  error rate  : 23.53%
[CHECK]  threshold   : 15.00%
[CHECK]  theory      : ~25% under full intercept-and-resend

[ALERT] 23.53% > 15.00%, eavesdropper detected, aborting


## Where Eve's errors come from

Split the sifted positions by whether Eve happened to pick Alice's basis.
- Eve right (50% of sifted) → she learns the bit, resends a state Bob measures correctly → no error.
- Eve wrong (50% of sifted) → her measurement randomises the state, Bob's outcome agrees with Alice's only 50% of the time → 50% error rate.

Net: $\tfrac{1}{2}\cdot 0 + \tfrac{1}{2}\cdot\tfrac{1}{2} = \tfrac{1}{4}$.

In [14]:
good_idx = [j for j, i in enumerate(keep) if eve_bases[i] == alice_bases[i]]
bad_idx  = [j for j, i in enumerate(keep) if eve_bases[i] != alice_bases[i]]

good_errs = sum(1 for j in good_idx if sifted_a[j] != sifted_b[j])
bad_errs  = sum(1 for j in bad_idx  if sifted_a[j] != sifted_b[j])

print("-" * 56)
print("Eve basis vs Alice basis breakdown (on sifted positions)")
print("-" * 56)
print(f"Eve right : {len(good_idx):3d} positions, {good_errs:2d} errors  "
      f"({0 if not good_idx else good_errs/len(good_idx):.1%})")
print(f"Eve wrong : {len(bad_idx):3d} positions, {bad_errs:2d} errors  "
      f"({0 if not bad_idx else bad_errs/len(bad_idx):.1%})")
print()
full_err = (good_errs + bad_errs) / len(keep) if keep else 0
print(f"Overall sifted-key error rate: {full_err:.2%}  (theory: ~25%)")

--------------------------------------------------------
Eve basis vs Alice basis breakdown (on sifted positions)
--------------------------------------------------------
Eve right :  24 positions,  0 errors  (0.0%)
Eve wrong :  29 positions, 20 errors  (69.0%)

Overall sifted-key error rate: 37.74%  (theory: ~25%)


## Attack 2: partial intercept

Eve attacks only a fraction $p$ of the qubits and lets the rest through untouched. On the attacked positions she still causes a 25% error rate; on the untouched positions there's no disturbance. Net sifted-key error rate is $\approx 0.25\,p$.

This lets us see how much eavesdropping a fixed threshold actually catches: at $\textsf{threshold} = 15\%$, anything with $p > 0.60$ should reliably trip it. Below that, Eve gets some information without detection, but also gets proportionally less of it.

In [15]:
def run_protocol(n: int, eve_prob: float) -> dict:
    """Run one full BB84 trial with Eve attacking each qubit independently with probability `eve_prob`.

    eve_prob = 0.0 -> no attacker; 1.0 -> full intercept-and-resend.
    Returns a dict of useful counts and the measured error rate on the check sample.
    """
    a_bits  = quantum_random_bits(n)
    a_bases = quantum_random_bits(n)
    e_bases = quantum_random_bits(n)
    b_bases = quantum_random_bits(n)

    # Decide which qubits Eve attacks, using quantum bits only.
    # Build a uniform [0, 1) value from 8 quantum bits per qubit, attack if < eve_prob.
    bits_per_decision = 8
    raw = quantum_random_bits(n * bits_per_decision)
    attack_mask = []
    for i in range(n):
        chunk = raw[i*bits_per_decision : (i+1)*bits_per_decision]
        u = sum(b << k for k, b in enumerate(chunk)) / (2**bits_per_decision)
        attack_mask.append(u < eve_prob)

    b_bits = []
    for i in range(n):
        sent = alice_encode(a_bits[i], a_bases[i])
        if attack_mask[i]:
            _, forwarded = eve_attack(sent, e_bases[i])
            b_bits.append(bob_measure(forwarded, b_bases[i]))
        else:
            b_bits.append(bob_measure(sent, b_bases[i]))

    keep = [i for i in range(n) if a_bases[i] == b_bases[i]]
    sa   = [a_bits[i] for i in keep]
    sb   = [b_bits[i] for i in keep]
    s    = max(1, len(sa) // 3)
    e    = sum(1 for x, y in zip(sa[:s], sb[:s]) if x != y)
    return {
        "sifted":     len(sa),
        "sample":     s,
        "errors":     e,
        "error_rate": e / s,
        "attacked":   sum(attack_mask),
    }

In [18]:
# Sweep p from 0 to 1
probs = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

print("-" * 64)
print(f"{'p':>6} {'attacked':>10} {'sifted':>8} {'errors':>8} {'rate':>10} {'detected':>10}")
print("-" * 64)
for p in probs:
    r = run_protocol(150, p)   # N=150 per sweep point
    detected = "yes" if r["error_rate"] > THRESHOLD else "no"
    print(f"{p:>6.2f} {r['attacked']:>10} {r['sifted']:>8} {r['errors']:>8} "
          f"{r['error_rate']:>10.2%} {detected:>10}")
print("-" * 64)
print(f"threshold: {THRESHOLD:.2%}   theory: error rate ≈ 0.25 * p")

----------------------------------------------------------------
     p   attacked   sifted   errors       rate   detected
----------------------------------------------------------------
  0.00          0       75        0      0.00%         no
  0.20         35       71        4     17.39%        yes
  0.40         67       71        1      4.35%         no
  0.60         91       74        1      4.17%         no
  0.80        124       73        7     29.17%        yes
  1.00        150       77        7     28.00%        yes
----------------------------------------------------------------
threshold: 15.00%   theory: error rate ≈ 0.25 * p


In [19]:
# What Eve actually learned
# On sifted positions, Eve only learns the bit when she happened to pick Alice's basis.
# At p=1 (full intercept), she gets ~half the sifted key correctly.
# At p<1, she only learns positions she actually attacked AND guessed the basis right.

leaked = [j for j, i in enumerate(keep)
          if eve_bases[i] == alice_bases[i]]
print(f"Eve learned {len(leaked)}/{len(keep)} sifted bits "
      f"({len(leaked)/len(keep):.1%}) — vs 0% with no attack")
print(f"(At full intercept: theory says ~50% of sifted key leaked)")

Eve learned 24/53 sifted bits (45.3%) — vs 0% with no attack
(At full intercept: theory says ~50% of sifted key leaked)


## With Eve vs without: side-by-side

Comparison of a clean run vs the full-attack run

In [17]:
clean = run_protocol(N, 0.0)

print("-" * 64)
print(f"{'scenario':<25} {'sifted':>8} {'error rate':>12} {'detected':>12}")
print("-" * 64)
print(f"{'no attacker':<25} {clean['sifted']:>8} {clean['error_rate']:>12.2%} "
      f"{('yes' if clean['error_rate'] > THRESHOLD else 'no'):>12}")
print(f"{'full intercept (p=1)':<25} {len(keep):>8} {err_rate:>12.2%} "
      f"{('yes' if err_rate > THRESHOLD else 'no'):>12}")
print(f"\nthreshold: {THRESHOLD:.2%}")

----------------------------------------------------------------
scenario                    sifted   error rate     detected
----------------------------------------------------------------
no attacker                     51        0.00%           no
full intercept (p=1)            53       23.53%          yes

threshold: 15.00%


## Summary

| Party | Role | What they do |
|-------|------|--------------|
| Alice | sender | encode random bits in random bases |
| Eve   | attacker | intercept, measure, resend a fresh qubit |
| Bob   | receiver | measure in random bases |

**Full intercept-and-resend** introduces a sifted-key error rate of ~25%:
- 50% of the time Eve's basis matches Alice's → no disturbance,
- 50% of the time it doesn't → 50% chance of error for Bob.

$\tfrac{1}{2}\cdot 0 + \tfrac{1}{2}\cdot\tfrac{1}{2} = \tfrac{1}{4}$.

**Partial intercept** scales linearly: attacking a fraction $p$ of qubits gives ~$0.25\,p$ error. At a 15% threshold, Eve has to stay below $p \approx 0.6$ to avoid detection, which also caps how much of the key she can learn. The sweep confirms this: the rate climbs smoothly with $p$ and crosses the threshold somewhere around $p = 0.6$.

The security argument is just the no-cloning theorem: Eve can't copy the qubit, so she has to measure, and measurement in the wrong basis is exactly what produces the disturbance that gives her away.